# Activité Pratique N°4 — Architecture Transformer
## Partie 1 — Étape 6 : Modèle de type BERT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Créer et tester un exemple basé sur un Transformer de type BERT : prédire un mot caché dans une séquence.

BERT est un **encodeur bidirectionnel** : pour remplacer `[MASK]`, il regarde le contexte **à gauche et à droite**.
On utilise `bert-base-multilingual-cased`, qui comprend le français.

```
Input : « Rabat est [MASK] du Maroc » ──► Bidirectional Encoder Representations from Transformers ──► Output : « capitale »
```

In [ ]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
modele = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased").to(device).eval()
print("Token de masque :", tokenizer.mask_token)

In [ ]:
def completer(phrase, k=5):
    phrase = phrase.replace("[MASK]", tokenizer.mask_token)
    entrees = tokenizer(phrase, return_tensors="pt").to(device)
    print(phrase)
    print("  tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))
    position = (entrees["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad():
        logits = modele(**entrees).logits[0, position]       # scores sur tout le vocabulaire
    top = torch.topk(torch.softmax(logits, dim=-1), k)
    for p, i in zip(top.values, top.indices):
        mot = tokenizer.decode([i.item()]).strip()
        print(f"  {p.item():6.2%}  {phrase.replace(tokenizer.mask_token, mot)}")
    print()


completer("Rabat est [MASK] du Maroc.")
completer("Rabat est la [MASK] du Maroc.")

### Le contexte de droite compte

In [ ]:
completer("Le [MASK] mange du fromage.")
completer("La [MASK] est connectée à l'ordinateur.")
completer("Casablanca est la plus grande [MASK] du Maroc.")

| | GPT | BERT |
|---|---|---|
| Architecture | Décodeur seul | Encodeur seul |
| Attention | Causale | Bidirectionnelle |
| Apprentissage | Prédire le token suivant | Prédire les tokens masqués |
| Usage | Génération, dialogue, code | Classification, extraction, compréhension |